In [1]:
import rustworkx as rx
from rustworkx.visualization import mpl_draw as draw_graph
import numpy as np
import matplotlib.pyplot as plt

import matplotlib
import warnings
warnings.simplefilter("ignore", UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.converters import circuit_to_dag,dag_to_circuit

import sys
sys.path.append("../")
from clapton.circuit_manipulation import transform_to_allowed_gates,qiskit_to_stim, modify_circuit, multi_angle_qaoa_circuit, transform_qiskit_to_stim,generate_qiskit_param_map
import testing_scripts.graphs_utils as graphs_utils
from testing_scripts.qaoa_utils import QAOASolver

In [2]:
n = 6
k=3

unweighted_G = graphs_utils.generate_k_regular_graph(num_vertices=n, k=k, weighted=False,seed=3)
# unweighted_G =graphs_utils.generate_random_complete_graph(num_vertices=n, weighted=False, seed=2)
# G = graphs_utils.generate_random_ego_graph(num_nodes=n,weighted=True,seed=2)
# G = graphs_utils.generate_random_erdos_renyi_graph(num_nodes=n,probability=0.4,weighted=True,seed=4)
# draw_graph(unweighted_G, node_size=600, with_labels=True)

In [3]:

unweighted_max_cut_paulis = graphs_utils.build_max_cut_paulis(unweighted_G)

unweighted_cost_hamiltonian = SparsePauliOp.from_list(unweighted_max_cut_paulis)
print("Cost Function Hamiltonian:", unweighted_cost_hamiltonian)
print(f"Number of terms : {len(unweighted_cost_hamiltonian)}")

Cost Function Hamiltonian: SparsePauliOp(['IIZZII', 'IIZIZI', 'IZIIZI', 'IIIZZI', 'ZIIIIZ', 'IZIIIZ', 'IIIZIZ', 'ZZIIII', 'ZIZIII'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])
Number of terms : 9


In [4]:
reps = 2
unweighted_circuit = QAOAAnsatz(cost_operator=unweighted_cost_hamiltonian, reps=reps)

In [5]:
unweighted_maxcut_qaoa = QAOASolver(unweighted_cost_hamiltonian,unweighted_circuit,sim_device="CPU")
unweighted_maxcut_qaoa.prepare_circuit()
unweighted_maxcut_qaoa.err=None

In [6]:
assert len(unweighted_maxcut_qaoa.stim_circ.gates) == unweighted_maxcut_qaoa.pcirc.size()

In [7]:
#Run CAFQA process
unweighted_maxcut_qaoa.run_cafqa(n_gens=25)

STARTING ROUND 0




started GA at id 1 with 8 procs

started GA at id 2 with 8 procs
GA parameters used for this experiment:
  num_generations=12
  num_parents_mating=20
  population_size=100
  num_genes=30
  parent_selection_type=tournament
  keep_parents=-1
  crossover_type=single_point
  mutation_type=adaptive
  crossover_probability=0.9
  mutation_probability=(0.25, 0.01)
  keep_elitism=5
started GA at id None with 8 procs

GA parameters used for this experiment:
  num_generations=12
  num_parents_mating=20
  population_size=100
  num_genes=30
  parent_selection_type=tournament
  keep_parents=-1
  crossover_type=single_point
  mutation_type=adaptive
  crossover_probability=0.9
  mutation_probability=(0.25, 0.01)
  keep_elitism=5

started GA at id 3 with 8 procs

GA parameters used for this experiment:
  num_generations=12
  num_parents_mating=20
  population_size=100
  num_genes=30
  parent_selection_type=tournament
  keep_parents=-1
  crossover_type=single_point
  mutation_type=adaptive
  crossover_p

In [8]:
unweighted_maxcut_qaoa.energy_best

np.float64(-3.0)

In [9]:
unweighted_maxcut_qaoa.evaluate_exact_energy()

Exact Energy from Eigensolver: -5.0


np.float64(-5.0)

In [10]:
ordered_params = [param.name for param in unweighted_maxcut_qaoa.pcirc.parameters]
angle_multipliers = [-np.pi/4 if 'gamma' in param else np.pi/4 for param in ordered_params]
cafqa_params = [param * (np.pi/2) for param, multiplier in zip(unweighted_maxcut_qaoa.ks_best, angle_multipliers)] #This has to be in the order we come across the gates.
unweighted_maxcut_qaoa.evaluate_energy(unweighted_maxcut_qaoa.pcirc,unweighted_cost_hamiltonian,cafqa_params)

array(-3.)

# Weighted Graph 

In [11]:
weighted_G = graphs_utils.generate_k_regular_graph(num_vertices=n, k=k, weighted=True,seed=3)
# weighted_G=graphs_utils.generate_random_complete_graph(num_vertices=n, weighted=True, seed=2)
# G = graphs_utils.generate_random_ego_graph(num_nodes=n,weighted=True,seed=2)
# G = graphs_utils.generate_random_erdos_renyi_graph(num_nodes=n,probability=0.4,weighted=True,seed=4)

In [12]:

weighted_max_cut_paulis = graphs_utils.build_max_cut_paulis(weighted_G)

weighted_cost_hamiltonian = SparsePauliOp.from_list(weighted_max_cut_paulis)
print("Cost Function Hamiltonian:", weighted_cost_hamiltonian)
print(f"Number of terms : {len(weighted_cost_hamiltonian)}")

Cost Function Hamiltonian: SparsePauliOp(['IZIIZI', 'ZIIZII', 'ZIZIII', 'IZIZII', 'ZIIIIZ', 'IIZIZI', 'IIIIZZ', 'IZZIII', 'IIIZIZ'],
              coeffs=[ 9.+0.j,  9.+0.j,  8.+0.j,  3.+0.j, 10.+0.j,  7.+0.j,  7.+0.j, 10.+0.j,
  8.+0.j])
Number of terms : 9


In [13]:
weighted_circuit = QAOAAnsatz(cost_operator=weighted_cost_hamiltonian, reps=reps)

In [14]:
weighted_maxcut_qaoa = QAOASolver(weighted_cost_hamiltonian,weighted_circuit,sim_device="CPU")
weighted_maxcut_qaoa.prepare_circuit()
weighted_maxcut_qaoa.err=None

In [15]:
ordered_params = [param.name for param in weighted_maxcut_qaoa.pcirc.parameters]
angle_multipliers = [-np.pi/4 if 'gamma' in param else np.pi/4 for param in ordered_params]
cafqa_params = [param * (np.pi/2) for param, multiplier in zip(unweighted_maxcut_qaoa.ks_best, angle_multipliers)] #This has to be in the order we come across the gates.
weighted_maxcut_qaoa.evaluate_energy(weighted_maxcut_qaoa.pcirc,weighted_cost_hamiltonian,cafqa_params)

array(10.)

In [16]:
weighted_maxcut_qaoa.evaluate_exact_energy()

Exact Energy from Eigensolver: -37.0


np.float64(-37.0)